---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 7 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Tim data engineering platform e-commerce meminta anda membangun **pipeline ETL produksi pertama** yang menggabungkan tiga sumber data sekaligus: transaksi (CSV), data produk (JSON), dan data ulasan pelanggan (CSV terpisah) — merepresentasikan kondisi nyata di mana data tersebar di berbagai sistem dengan format berbeda-beda.

### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan **tiga sumber data** sekaligus.

In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


### Instruksi Pengerjaan

Buat notebook baru **`Tugas6_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, lalu bangun pipeline ETL lengkap mengikuti struktur **Extract → Transform → Load**:

---

**A. EXTRACT** *(bobot 15%)*

Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan `printSchema()` masing-masing.

**B. TRANSFORM — Penggabungan** *(bobot 25%)*

Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk). Gunakan **`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan **`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).

**C. TRANSFORM — Penanganan Data Kosong & Pengayaan** *(bobot 20%)*

- Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `salah satu join yang tepat` — isi nilai kosong tersebut dengan angka **0** menggunakan `salah satu function`, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
- Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu)`, **sebelum** langkah `na.fill()` di atas).

**D. LOAD** *(bobot 25%)*

Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`. Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.

**E. Insight Akhir** *(bobot 15%)*

Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas6_[NPM]_[Nama Lengkap].ipynb` melalui Asprak, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan Hadoop aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Extract | Ketiga sumber berhasil dibaca dengan skema yang benar | 15% |
| B. Transform - Join | Kedua join (left & inner) diterapkan dengan tepat sesuai kebutuhan masing-masing | 25% |
| C. Transform - Data Quality | Missing value tertangani logis; kolom `ada_ulasan` benar | 20% |
| D. Load | Data tersimpan ke HDFS sebagai Parquet dengan partisi yang benar & terverifikasi | 25% |
| E. Insight Akhir | Analisis tepat & interpretasi bisnis relevan | 15% |

### SparkSession Baru

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/24 19:42:11 WARN Utils: Your hostname, kyadevi resolves to a loopback address: 127.0.1.1; using 192.168.1.16 instead (on interface wlp0s20f3)
26/09/24 19:42:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 19:42:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 19:42:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/24 19:42:12 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


SparkSession siap. Versi Spark: 3.5.9


**A. EXTRACT** *(bobot 15%)*

Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan `printSchema()` masing-masing.

In [57]:
print("<<< TRANSAKSI >>>")
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
print("Transaksi:", df_transaksi.count(), "baris")
df_transaksi.printSchema()

print("<<< PRODUK >>>")
df_produk = spark.read.json("tugas6_produk.json")
print("Produk:", df_produk.count(), "baris")
df_produk.printSchema()

print("<<< ULASAN >>>")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)
print("Ulasan:", df_ulasan.count(), "baris")
df_ulasan.printSchema()

<<< TRANSAKSI >>>
Transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

<<< PRODUK >>>
Produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

<<< ULASAN >>>
Ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



**B. TRANSFORM — Penggabungan** *(bobot 25%)*

Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk). Gunakan **`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan **`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).

In [58]:
# Left Join: Menggabungkan Transaksi dengan Ulasan menggunakan order_id
df_join1 = df_transaksi.join(df_ulasan, on="order_id", how="left")

# Inner Join: Menggabungkan df_join1 dengan produk menggunakan product_id
df_join2 = df_join1.join(df_produk, on="product_id", how="inner")

### Menambahkan kolom total pendapatan
df_join2 = df_join2.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga")
)


df_join2 = df_join2.select(
    "order_id", "product_id", "nama_produk", 
    "unit_terjual", "kategori", "harga", "rating", "total_pendapatan"
)

df_join2.show(5)



+--------+----------+-----------+------------+------------+------+------+----------------+
|order_id|product_id|nama_produk|unit_terjual|    kategori| harga|rating|total_pendapatan|
+--------+----------+-----------+------------+------------+------+------+----------------+
|     TX0|        21|  Produk-21|           2|  Elektronik| 25000|     1|           50000|
|     TX1|         8|   Produk-8|           6|     Fashion|250000|  NULL|         1500000|
|     TX2|        25|  Produk-25|           4|  Elektronik| 25000|     2|          100000|
|     TX3|         3|   Produk-3|           4|     Fashion| 75000|     5|          300000|
|     TX4|        19|  Produk-19|           6|Rumah Tangga| 75000|  NULL|          450000|
+--------+----------+-----------+------------+------------+------+------+----------------+
only showing top 5 rows



**C. TRANSFORM — Penanganan Data Kosong & Pengayaan** *(bobot 20%)*

- Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `salah satu join yang tepat` — isi nilai kosong tersebut dengan angka **0** menggunakan `salah satu function`, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
- Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu)`, **sebelum** langkah `na.fill()` di atas).

In [61]:
# Tambahkan kolom ada ulasan sebelum na.fill()
df_rating = df_join2.withColumn(
    "ada_ulasan?",
    when(col("rating").isNotNull(), True).otherwise(False)
)

# Isi rating null dengan 0
df_rating = df_rating.na.fill({"rating": 0})

df_rating.select("order_id", "rating", "ada_ulasan?").show(10)

+--------+------+-----------+
|order_id|rating|ada_ulasan?|
+--------+------+-----------+
|     TX0|     1|       true|
|     TX1|     0|      false|
|     TX2|     2|       true|
|     TX3|     5|       true|
|     TX4|     0|      false|
|     TX5|     5|       true|
|     TX6|     4|       true|
|     TX7|     0|      false|
|     TX8|     5|       true|
|     TX9|     0|      false|
+--------+------+-----------+
only showing top 10 rows



**D. LOAD** *(bobot 25%)*

Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`. Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.

In [46]:
!hdfs dfs -mkdir -p /user/kyadevi/tugas6/hasil_etl

df_rating.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/kyadevi/tugas6/hasil_etl"
)

print("Pipeline ETL selesai — hasil tersimpan di HDFS.")
!hdfs dfs -ls /user/kyadevi/tugas6/hasil_etl

Pipeline ETL selesai — hasil tersimpan di HDFS.
Found 6 items
-rw-r--r--   3 kyadevi supergroup          0 2026-09-24 22:01 /user/kyadevi/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - kyadevi supergroup          0 2026-09-24 22:01 /user/kyadevi/tugas6/hasil_etl/kategori=Elektronik
drwxr-xr-x   - kyadevi supergroup          0 2026-09-24 22:01 /user/kyadevi/tugas6/hasil_etl/kategori=Fashion
drwxr-xr-x   - kyadevi supergroup          0 2026-09-24 22:01 /user/kyadevi/tugas6/hasil_etl/kategori=Kesehatan
drwxr-xr-x   - kyadevi supergroup          0 2026-09-24 22:01 /user/kyadevi/tugas6/hasil_etl/kategori=Makanan
drwxr-xr-x   - kyadevi supergroup          0 2026-09-24 22:01 /user/kyadevi/tugas6/hasil_etl/kategori=Rumah Tangga


**E. Insight Akhir** *(bobot 15%)*

Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?


In [66]:
# Hitung total transaksi & jumlah yang ada ulasan per kategori
insight = df_rating.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    spark_sum(when(col("ada_ulasan?") == True, 1).otherwise(0)).alias("transaksi_berulasan")
)

# Hitung persentase
insight = insight.withColumn(
    "persentase_berulasan",
    spark_round((col("transaksi_berulasan") / col("total_transaksi")) * 100, 2)
)

# Urutkan dari persentase terkecil
insight.orderBy("persentase_berulasan").show()

+------------+---------------+-------------------+--------------------+
|    kategori|total_transaksi|transaksi_berulasan|persentase_berulasan|
+------------+---------------+-------------------+--------------------+
|     Makanan|            536|                369|               68.84|
|   Kesehatan|            961|                662|               68.89|
|     Fashion|           1035|                726|               70.14|
|Rumah Tangga|            815|                575|               70.55|
|  Elektronik|           1653|               1168|               70.66|
+------------+---------------+-------------------+--------------------+



    Kategori Makanan memiliki persentase transaksi berulasan paling rendah, yang berarti pelanggan yang membeli produk Makanan paling jarang meninggalkan ulasan dibandingkan kategori lain. Hal ini penting bagi tim marketing karena rendahnya ulasan dapat menandakan keterlibatan pelanggan yang rendah atau kepuasan yang tidak terungkap — sehingga tim marketing perlu mendorong pelanggan kategori ini untuk memberi feedback, misalnya melalui program poin ulasan atau diskon pembelian berikutnya. Tanpa ulasan yang cukup, tim marketing akan kesulitan mengevaluasi kualitas produk Makanan dan mengidentifikasi peluang perbaikan.